# Data Preprocessing Tools

## Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

In [2]:
pwd

'C:\\Udemy\\machinelearning-june-2026\\Machine Learning A-Z (Codes and Datasets)-20260626T175232Z-3-001\\Part 1 - Data Preprocessing\\Section 2 -------------------- Part 1 - Data Preprocessing --------------------\\Python'

In [3]:
dataset = pd.read_csv('Data.csv') # Loading the dataset

In [4]:
dataset.describe()

,Age,Salary
count,9.000000,9.000000
mean,38.777778,63777.777778
std,7.693793,12265.579662
min,27.000000,48000.000000
25%,35.000000,54000.000000
50%,38.000000,61000.000000
75%,44.000000,72000.000000
max,50.000000,83000.000000


In [5]:
dataset.head()

,Country,Age,Salary,Purchased
0,France,44.0,72000.0,No
1,Spain,27.0,48000.0,Yes
2,Germany,30.0,54000.0,No
3,Spain,38.0,61000.0,No
4,Germany,40.0,NaN,Yes


In [6]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Country    10 non-null     str    
 1   Age        9 non-null      float64
 2   Salary     9 non-null      float64
 3   Purchased  10 non-null     str    
dtypes: float64(2), str(2)
memory usage: 452.0 bytes


In [4]:


# the independent variables, are the variables containing some information with which you can predict what you want to predict, which is called the dependent variable.
# Creating the matrix of features (X) and the dependent variable vector (y)
X = dataset.iloc[:, :-1].values # independent variables
y = dataset.iloc[:, -1].values # dependent variable

In [5]:
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 nan]
 ['France' 35.0 58000.0]
 ['Spain' nan 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


In [6]:
print(y)

<StringArray>
['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes']
Length: 10, dtype: str


## Taking care of missing data

In [7]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean') # Replacing missing values with the mean
imputer.fit(X[:, 1:3]) # Fitting the imputer to the columns with missing data (columns 1 and 2)
# I recommend on a general rule to select all the numerical columns because in your career you will actually work with huge data sets and you won't be able to see where the missing values are.

X[:, 1:3] = imputer.transform(X[:, 1:3])

In [8]:
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 63777.77777777778]
 ['France' 35.0 58000.0]
 ['Spain' 38.77777777777778 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


## Encoding categorical data

### Encoding the Independent Variable

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [ ]:
print(X)

### Encoding the Dependent Variable

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
print(y)

## Splitting the dataset into the Training set and Test set

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1) # Splitting the dataset into the training set and the test set
# where `test_size` is the proportion of the dataset to include in the test split (0.2 means 20% of the data will be used for testing), and `random_state` is a seed value that ensures reproducibility of the split.
# random_state is used to ensure that the split of the dataset into training and test sets is reproducible. When you set a specific random_state value, it initializes the random number generator in a way that produces the same sequence of random numbers each time you run the code. This means that every time you run the code with the same random_state, you'll get the same split of the dataset into training and test sets.

In [ ]:
print(X_train)

In [ ]:
print(X_test)

In [ ]:
print(y_train)

In [ ]:
print(y_test)

## Feature Scaling

Feature scaling should be applied **after** splitting the dataset into the training set and the test set. This is the only correct approach because the test set must remain completely separate from the training process.

First, let us clarify what each step does. Splitting the dataset creates two separate sets: a training set and a test set. The training set contains the existing observations used to train the machine learning model. The test set is used only after training, to evaluate how well the model performs on observations it has never seen before. In practice, the test set represents future data that the model will receive after it has been deployed.

Feature scaling rescales the input features so that they are on comparable scales. This prevents features with larger numerical ranges from dominating features with smaller ranges. Without scaling, the model may give too much importance to one feature and too little importance to another simply because their numerical values are measured on different scales.

The reason scaling must happen after the split is that feature scaling calculates statistics from the data, such as the mean and standard deviation. If we apply feature scaling before splitting the dataset, those statistics are computed using all observations, including the observations that will later become part of the test set.

That would cause **data leakage**. The test set is supposed to behave like brand-new future data, so it should not influence any part of the training process. If information from the test set is used to calculate the scaling parameters, then the model has indirectly learned something from data it was not supposed to see.

The correct workflow is therefore:

1. Split the dataset into the training set and the test set.
2. Fit the scaler only on the training set.
3. Transform the training set with that fitted scaler.
4. Transform the test set with the same fitted scaler.

This keeps the test set truly independent and gives a more realistic evaluation of how the model will perform on new data.

Now that we understand the **what** and the **why**, let us look at the **how**: how do we actually apply feature scaling?

There are two main feature scaling techniques: **standardization** and **normalization**.

**Standardization** subtracts the mean of a feature from each value, then divides the result by the standard deviation. The standard deviation is the square root of the variance. After standardization, the values of each feature are usually centered around 0 and often fall roughly between -3 and +3.

**Normalization** subtracts the minimum value of a feature from each value, then divides the result by the difference between the maximum and minimum values. This transforms the values so that they fall between 0 and 1.

A common question is whether we should use standardization or normalization. A practical rule is this: normalization can be useful when most of the features follow a normal distribution, but standardization works well in most situations. For that reason, we will use **standardization** here.

Since feature scaling must be applied after the split, we do not apply it to the original matrix of features `X`. Instead, we apply it to the two feature matrices created by the split: `X_train` and `X_test`.

The scaler is fitted only on `X_train`. This means it calculates the mean and standard deviation using only the training data. Then we use those same training-set statistics to transform both `X_train` and `X_test`.

This is why the code uses `fit_transform` on `X_train`, but only `transform` on `X_test`. We are allowed to learn the scaling parameters from the training set, but we are not allowed to learn anything from the test set. The test set must remain new, unseen data until the model is evaluated.

Another important question is whether we should apply feature scaling to the dummy variables created by one-hot encoding.

The answer is **no**. The goal of feature scaling is to put features on a comparable numerical scale. Dummy variables already take values of either 0 or 1, so they are already within a small range. Standardizing them would not add useful information.

More importantly, scaling dummy variables makes them harder to interpret. Before scaling, a tuple such as `[1, 0, 0]` clearly represents one category, while `[0, 0, 1]` represents another. After standardization, those clean 0 and 1 values become transformed numerical values, and the original meaning of each dummy-variable pattern becomes much less clear.

In practice, scaling dummy variables rarely improves model performance enough to justify losing that interpretability. The important columns to scale here are the numerical features, such as `Age` and `Salary`, because they are measured on very different ranges.

So in this dataset, we leave the dummy-variable columns unchanged and apply standardization only to the numerical columns.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train[:, 3:] = sc.fit_transform(X_train[:, 3:]) # Fit and transform the training set
X_test[:, 3:] = sc.transform(X_test[:, 3:]) # Transform the test set using the same scaler

In [ ]:
print(X_train)

In [ ]:
print(X_test)

This final step applies feature scaling to the numerical columns of the training set and then applies the same transformation to the numerical columns of the test set.

The key detail is that `X_test` uses `transform`, not `fit_transform`. The test set represents new data that the model receives later, so we must not fit a new scaler on it. If we used `fit_transform` on `X_test`, we would calculate a new mean and standard deviation from the test data, creating a different scaling rule from the one used during training.

Instead, the scaler is fitted once on `X_train`. The same scaler is then used to transform `X_test`, so the test data is prepared in a way that is consistent with the data used to train the model. This consistency is essential because `X_test` will later be passed into the model's `predict` method.

When we print `X_train` and `X_test`, the dummy-variable columns remain unchanged as 0s and 1s. The numerical columns, `Age` and `Salary`, are transformed so that they are now on a similar scale, typically around -3 to +3. In this small dataset, the scaled values are closer to about -2 to +2, which is still exactly what we want.

At this point, the data preprocessing toolkit is complete. The dataset has been cleaned, encoded, split, and scaled, so it is ready to be used for training machine learning models.